# 02 - Trait space in 2-D: Fig. 2b (PC1-PC2) and Fig. 2c (PC2-PC3)

**Purpose.** Draw the two-dimensional biplots of the functional trait space. The 10,000-point sample
is projected onto the published PCA axes; points are coloured by NLCD land-cover class, contours
enclose 50 % and 99 % of the points, and arrows show the trait loadings coloured by functional group.

**Inputs.** `../data/pca_model/pca_sample_points.csv`, `../data/pca_model/scaler_rp_10traits.pkl`,
`../data/pca_model/PCA_model_rp_sa_10traits.pkl` (the model fitted in notebook 01).

**Outputs.** `./results/fig2b_trait_space_PC1_PC2.png` (Fig. 2b) and
`./results/fig2c_trait_space_PC2_PC3.png` (Fig. 2c).

Run with the working directory set to this folder.

In [ ]:
import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
import matplotlib.patheffects as pe
from matplotlib import cm
from matplotlib.lines import Line2D
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist, squareform

DATA_DIR = '../data/pca_model'
OUT_DIR = './results'
os.makedirs(OUT_DIR, exist_ok=True)

# the ten traits entering the PCA (column names after renaming in the next cell)
trait_list = ['Carbon', 'Cellulose', 'Chlorophyll a + b', 'EWT', 'Lignin', 'Nitrogen',
              'NSC', 'Phenolics', 'SLA', 'Canopy Height']

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, 'pca_sample_points.csv'))
df.rename(columns={'ChlorophyllsArea': 'Chlorophyll a + b', 'canopy_height': 'Canopy Height'}, inplace=True)
# canopy height is an integer raster value; +1 avoids log(0) for zero-height pixels
df['Canopy Height'] = df['Canopy Height'] + 1

In [ ]:
df = df.dropna()

In [ ]:
# natural vegetation only: drop NLCD 81 (pasture/hay) and 82 (cultivated crops)
df = df.query('nlcd_class != 81 and nlcd_class != 82')

In [ ]:
# random subsample of 10,000 points (fixed seed) used for the PCA and the trait-space figures
df = df.sample(n=10000, random_state=1)

In [ ]:
nlcd_dict = {0: 'Open Water', 11: 'Developed, Open Space', 12: 'Developed, Low Intensity',
             21: 'Developed, Medium Intensity',
             22: 'Developed, High Intensity', 23: 'Developed, Open Space with Buildings',
             24: 'Developed, Open Space with Roads',
             31: 'Barren Land (Rock/Sand/Clay)', 41: 'Deciduous Forest', 42: 'Evergreen Forest', 43: 'Mixed Forest',
             51: 'Dwarf Scrub', 52: 'Shrub/Scrub', 71: 'Grassland/Herbaceous', 72: 'Sedge/Herbaceous', 73: 'Lichens',
             74: 'Moss', 81: 'Pasture/Hay', 82: 'Cultivated Crops', 90: 'Woody Wetlands',
             95: 'Emergent Herbaceous Wetlands'}
df['nlcd'] = df['nlcd_class'].map(nlcd_dict)

# Trait functional groups

Used to colour the trait labels in the biplots.

In [ ]:
# functional group of each trait (colours the trait labels in the biplots)
function = {'Canopy Height': 'Plant/Leaf structure', 'Carbon': 'Plant/Leaf structure',
            'EWT': 'Plant/Leaf structure',
            'Nitrogen': 'Light capture and growth', 'NSC': 'Light capture and growth',
            'Chlorophyll a + b': 'Light capture and growth', 'SLA': 'Light capture and growth',
            'Phenolics': 'Defense', 'Cellulose': 'Defense', 'Lignin': 'Defense'}

function_color = {'Plant/Leaf structure': '#A626A4', 'Light capture and growth': '#009E73',
                  'Defense': '#4053D3'}

# Spatially de-trended PCA helpers

The functions used in notebook 01 to fit the model, kept here for reference; the model itself is loaded from the archive in the next cell.

In [ ]:
def calculate_moran_i(data, weights):
    """Moran's I statistic of a 1-D variable under a spatial weight matrix."""
    mean = np.mean(data)
    n = len(data)
    numerator = np.sum(weights * (data[:, np.newaxis] - mean) * (data[np.newaxis, :] - mean))
    denominator = np.sum((data - mean) ** 2)
    return (n / np.sum(weights)) * (numerator / denominator)


def remove_spatial_autocorrelation(data, coordinates):
    """Remove spatial autocorrelation from every column of `data`.

    For each column a spatial-lag regression (y ~ 1 + Wy, W = inverse-distance weights)
    is fitted and the residuals are returned.
    """
    # spatial weight matrix: inverse distance; 1 on the diagonal avoids division by zero
    distances = pdist(coordinates)
    dist_matrix = squareform(distances)
    weights = 1 / (dist_matrix + np.eye(dist_matrix.shape[0]))

    corrected_data = np.zeros_like(data)
    for i in range(data.shape[1]):
        y = data[:, i]
        W = weights

        # spatial lag of y
        spatial_lag = W.dot(y) / W.sum(axis=1)

        # spatial regression
        X = np.column_stack((np.ones_like(y), spatial_lag))
        beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)

        # residuals = data with the spatial autocorrelation removed
        corrected_data[:, i] = y - X.dot(beta)

    return corrected_data


def spatial_pca(data, coordinates, n_components=10):
    """PCA of spatially de-trended data (see remove_spatial_autocorrelation)."""
    corrected_data = remove_spatial_autocorrelation(data, coordinates)

    pca = PCA(n_components=n_components)
    pca.fit(corrected_data)

    return pca, pca.explained_variance_ratio_

# Load the published scaler and PCA (fitted in notebook 01)

In [ ]:
scaler_all = pickle.load(open(os.path.join(DATA_DIR, 'scaler_rp_10traits.pkl'), 'rb'))
pca_all = pickle.load(open(os.path.join(DATA_DIR, 'PCA_model_rp_sa_10traits.pkl'), 'rb'))
print('explained variance ratio:', np.round(pca_all.explained_variance_ratio_, 4))

In [ ]:
# project the sample onto the published axes: log -> standardise -> PCA
# (pca_all.components_ already carry the sign flip applied in notebook 01)
X = np.log(df[trait_list])
X_scale = scaler_all.transform(X)
X_scale_reduced = pca_all.transform(X_scale)

X_scale_reduced_12 = pd.DataFrame(X_scale_reduced[:, :2], index=df.index, columns=['PC1', 'PC2'])
X_scale_reduced_12['nlcd'] = df['nlcd']

# Fig. 2b and Fig. 2c

Trait names are bold with a white halo so they stay legible over the NLCD point colours.
`SCHEME` selects the label palette; `'orig_bold'` is the published one, `'dark'` is an
alternative with colours further away from the point colours.

In [ ]:
plt.style.use('default')
sns.set_style('whitegrid')
plt.style.use('bmh')
sns.set_context('paper')
plt.rcParams['font.family'] = ['Helvetica', 'Arial', 'DejaVu Sans']

# --- trait-name label style -------------------------------------------------
SCHEME = 'orig_bold'  # 'orig_bold' or 'dark'

label_palettes = {
    # original three colours, unchanged
    'orig_bold': {'Plant/Leaf structure': '#A626A4', 'Light capture and growth': '#009E73',
                  'Defense': '#4053D3'},
    # darker: deep crimson / very dark green / dark brown, further from the point colours
    'dark': {'Plant/Leaf structure': '#C51B7D', 'Light capture and growth': '#00441B',
             'Defense': '#7F3B08'},
}
function_color_v2 = label_palettes[SCHEME]
label_weight = 'bold'
label_halo = [pe.withStroke(linewidth=2.5, foreground='white')]
# ----------------------------------------------------------------------------

norm = matplotlib.colors.Normalize(vmin=0, vmax=6)
rgba = cm.gist_ncar([norm(0), norm(1), norm(2), norm(3), norm(4), norm(5), norm(6)])

# rgba to list of hex colours, one per NLCD class
color_list = []
for i in range(rgba.shape[0]):
    color_list.append(matplotlib.colors.rgb2hex(rgba[i, :3]))

color_list[2] = '#006E00'
color_list[6] = '#7F7F7F'


def PCA_2D_plot(score, coeff, labels=None, hue=None):
    fig, ax = plt.subplots(dpi=300, figsize=(5.5 / 1.2, 4.5 / 1.2))
    xs = score.iloc[:, 0]
    ys = score.iloc[:, 1]
    n = coeff.shape[0]

    sns.kdeplot(x=xs, y=ys, ax=ax,
                levels=[0.01, 0.5],
                **{'linewidths': 1, 'linestyles': '-'},
                colors=['#EBCB8B', '#FF0000']
                )

    sns.scatterplot(x=xs, y=ys, data=score, ax=ax, s=1, hue=hue,
                    **{'edgecolor': 'none', 'alpha': 0.8}, palette=color_list,
                    hue_order=['Deciduous Forest', 'Mixed Forest', 'Evergreen Forest', 'Shrub/Scrub',
                               'Grassland/Herbaceous', 'Woody Wetlands'])

    arrow_color = '#05445E'
    enlarge = 9
    for i in range(n):
        if labels[i] in ['Phenolics']:
            ax.annotate(labels[i], xy=(0, 0), xytext=(coeff[i, 0] * enlarge, coeff[i, 1] * enlarge),
                        color=[0, 0, 0, 0],
                        arrowprops=dict(arrowstyle="<-", lw=1, color=arrow_color, linestyle='--')
                        , va='center', ha='center'
                        )
            ax.annotate(labels[i], xy=(coeff[i, 0] * enlarge, coeff[i, 1] * enlarge)
                        , va='center', ha='left', color=function_color_v2[function[labels[i]]], fontweight=label_weight,
                        path_effects=label_halo
                        )
        elif labels[i] in ['Chlorophyll a + b']:
            ax.annotate(labels[i], xy=(0, 0), xytext=(coeff[i, 0] * enlarge, coeff[i, 1] * enlarge),
                        color=[0, 0, 0, 0],
                        arrowprops=dict(arrowstyle="<-", lw=1, color=arrow_color, linestyle='--')
                        , va='center', ha='center'
                        )
            ax.annotate(labels[i], xy=(coeff[i, 0] * enlarge, coeff[i, 1] * enlarge)
                        , va='center', ha='right', color=function_color_v2[function[labels[i]]], fontweight=label_weight,
                        path_effects=label_halo
                        )
        elif labels[i] in ['Canopy Height']:
            ax.annotate(labels[i], xy=(0, 0), xytext=(coeff[i, 0] * 15, coeff[i, 1] * 15),
                        color=[0, 0, 0, 0],
                        arrowprops=dict(arrowstyle="<-", lw=1, color=arrow_color, linestyle='--')
                        , va='center', ha='center'
                        )
            ax.annotate(labels[i], xy=(coeff[i, 0] * 15, coeff[i, 1] * 15)
                        , va='center', ha='right', color=function_color_v2[function[labels[i]]], fontweight=label_weight,
                        path_effects=label_halo
                        )

        else:
            ax.annotate(labels[i], xy=(0, 0), xytext=(coeff[i, 0] * enlarge, coeff[i, 1] * enlarge),
                        color=function_color_v2[function[labels[i]]], fontweight=label_weight,
                        arrowprops=dict(arrowstyle="<-", lw=1, color=arrow_color, linestyle='--')
                        , va='center', ha='center'
                        , path_effects=label_halo
                        )

    # legend at left upper corner
    ax.legend(loc='upper left', bbox_to_anchor=(-0.01, 1.03), ncol=1, markerscale=3, frameon=False, fontsize=8,
              handletextpad=0.4)
    ax.grid(False)

    plt.xlabel("PC{} ({}%)".format(1, round(pca_all.explained_variance_ratio_[0] * 100, 1)))
    plt.ylabel("PC{} ({}%)".format(2, round(pca_all.explained_variance_ratio_[1] * 100, 1)))
    ax.set_facecolor('1')
    for spine in ax.spines.values():
        spine.set_edgecolor('k')
        spine.set_linewidth(1.5)
    return fig, ax


fig, ax = PCA_2D_plot(X_scale_reduced_12, np.transpose(pca_all.components_), labels=trait_list,
                      hue='nlcd')

# remove top and right border
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# second (invisible) axis that only carries the contour legend
ax2 = ax.twinx()
ax2.set_frame_on(False)
ax2.axis('off')
ax.tick_params(axis='both', left=False, top=False, right=False, bottom=False)

legend_elements = [Line2D([0], [0], color='#EBCB8B', lw=1, label='99%'),
                   Line2D([0], [0], color='#FF0000', lw=1, label='50%')]
ax2.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, 0), ncol=1,
           markerscale=3, frameon=False, fontsize='small')

ax.text(0.03, 0.15, 'Plant/Leaf structure', color=function_color_v2['Plant/Leaf structure'], fontsize
='small', fontweight=label_weight, ha='left', va='top', transform=ax.transAxes, path_effects=label_halo)
ax.text(0.03, 0.10, 'Light capture and growth', color=function_color_v2['Light capture and growth'], fontsize
='small', fontweight=label_weight, ha='left', va='top', transform=ax.transAxes, path_effects=label_halo)
ax.text(0.03, 0.05, 'Defense', color=function_color_v2['Defense'], fontsize
='small', fontweight=label_weight, ha='left', va='top', transform=ax.transAxes, path_effects=label_halo)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig2b_trait_space_PC1_PC2.png'), dpi=1000, bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
plt.style.use('default')
sns.set_style('whitegrid')
plt.style.use('bmh')
sns.set_context('paper')
plt.rcParams['font.family'] = ['Helvetica', 'Arial', 'DejaVu Sans']

# --- trait-name label style -------------------------------------------------
SCHEME = 'orig_bold'  # 'orig_bold' or 'dark'

label_palettes = {
    # original three colours, unchanged
    'orig_bold': {'Plant/Leaf structure': '#A626A4', 'Light capture and growth': '#009E73',
                  'Defense': '#4053D3'},
    # darker: deep crimson / very dark green / dark brown, further from the point colours
    'dark': {'Plant/Leaf structure': '#C51B7D', 'Light capture and growth': '#00441B',
             'Defense': '#7F3B08'},
}
function_color_v2 = label_palettes[SCHEME]
label_weight = 'bold'
label_halo = [pe.withStroke(linewidth=2.5, foreground='white')]
# ----------------------------------------------------------------------------

norm = matplotlib.colors.Normalize(vmin=0, vmax=6)
rgba = cm.gist_ncar([norm(0), norm(1), norm(2), norm(3), norm(4), norm(5), norm(6)])

# rgba to list of hex colours, one per NLCD class
color_list = []
for i in range(rgba.shape[0]):
    color_list.append(matplotlib.colors.rgb2hex(rgba[i, :3]))

color_list[2] = '#006E00'
color_list[6] = '#7F7F7F'


def PCA_2D_plot(score, coeff, labels=None, hue=None):
    fig, ax = plt.subplots(dpi=300, figsize=(5.5 / 1.2, 4.5 / 1.2))
    xs = score.iloc[:, 0]
    ys = score.iloc[:, 1]
    n = coeff.shape[0]

    sns.kdeplot(x=xs, y=ys, ax=ax,
                levels=[0.01, 0.5],
                **{'linewidths': 1, 'linestyles': '-'},
                colors=['#EBCB8B', '#FF0000']
                )

    sns.scatterplot(x=xs, y=ys, data=score, ax=ax, s=1, hue=hue,
                    **{'edgecolor': 'none', 'alpha': 0.8}, palette=color_list,
                    hue_order=['Deciduous Forest', 'Mixed Forest', 'Evergreen Forest', 'Shrub/Scrub',
                               'Grassland/Herbaceous', 'Woody Wetlands'])

    arrow_color = '#05445E'
    enlarge = 6
    for i in range(n):
        if labels[i] in ['Phenolics']:
            ax.annotate(labels[i], xy=(0, 0), xytext=(coeff[i, 0] * enlarge, coeff[i, 1] * enlarge),
                        color=[0, 0, 0, 0],
                        arrowprops=dict(arrowstyle="<-", lw=1, color=arrow_color, linestyle='--')
                        , va='center', ha='center'
                        )
            ax.annotate(labels[i], xy=(coeff[i, 0] * enlarge, coeff[i, 1] * enlarge)
                        , va='center', ha='left', color=function_color_v2[function[labels[i]]], fontweight=label_weight,
                        path_effects=label_halo
                        )
        else:
            ax.annotate(labels[i], xy=(0, 0), xytext=(coeff[i, 0] * enlarge, coeff[i, 1] * enlarge),
                        color=function_color_v2[function[labels[i]]], fontweight=label_weight,
                        arrowprops=dict(arrowstyle="<-", lw=1, color=arrow_color, linestyle='--')
                        , va='center', ha='center'
                        , path_effects=label_halo
                        )

    # legend at left upper corner
    ax.legend(loc='upper left', bbox_to_anchor=(-0.01, 1.03), ncol=1, markerscale=3, frameon=False, fontsize=8,
              handletextpad=0.4)
    ax.set_xlim(-7, 5.5)
    ax.grid(False)

    plt.xlabel("PC{} ({}%)".format(2, round(pca_all.explained_variance_ratio_[1] * 100, 1)))
    plt.ylabel("PC{} ({}%)".format(3, round(pca_all.explained_variance_ratio_[2] * 100, 1)))
    ax.set_facecolor('1')
    for spine in ax.spines.values():
        spine.set_edgecolor('k')
        spine.set_linewidth(1.5)
    return fig, ax


X_scale_reduced_23 = pd.DataFrame(X_scale_reduced[:, 1:3], index=df.index, columns=['PC2', 'PC3'])
X_scale_reduced_23['nlcd'] = df['nlcd']
fig, ax = PCA_2D_plot(X_scale_reduced_23, np.transpose(pca_all.components_[1:3]), labels=trait_list,
                      hue='nlcd')

# remove top and right border
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# second (invisible) axis that only carries the contour legend
ax2 = ax.twinx()
ax2.set_frame_on(False)
ax2.axis('off')
ax.tick_params(axis='both', left=False, top=False, right=False, bottom=False)

legend_elements = [Line2D([0], [0], color='#EBCB8B', lw=1, label='99%'),
                   Line2D([0], [0], color='#FF0000', lw=1, label='50%')]
ax2.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, 0), ncol=1,
           markerscale=3, frameon=False, fontsize='small')

ax.text(0.03, 0.15, 'Plant/Leaf structure', color=function_color_v2['Plant/Leaf structure'], fontsize
='small', fontweight=label_weight, ha='left', va='top', transform=ax.transAxes, path_effects=label_halo)
ax.text(0.03, 0.10, 'Light capture and growth', color=function_color_v2['Light capture and growth'], fontsize
='small', fontweight=label_weight, ha='left', va='top', transform=ax.transAxes, path_effects=label_halo)
ax.text(0.03, 0.05, 'Defense', color=function_color_v2['Defense'], fontsize
='small', fontweight=label_weight, ha='left', va='top', transform=ax.transAxes, path_effects=label_halo)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig2c_trait_space_PC2_PC3.png'), dpi=1000, bbox_inches='tight')
plt.show()
plt.close(fig)